# D-algebraic functions expansion

In [1]:
import sys
sys.path.insert(0, "..") # dalgebra is here
from dalgebra import *
from dalgebra.pseries.laurent import *

%display latex

I want to think about what to do to detect possible orders for Laurent series expansions around 0 of solutions of D-algebraic equations. 

I am not fuly sure on all the steps but I have an intuition on how to proceed.

In order to work properly, I am going to need at least three examples:
* A linear example: things are easy here and a polynomial (indicial polynomial) can be computed.
* A purely D-algebraic case:
  - With guaranteed highest order monomial with its highest derivative
  - With highest order monomial without its highest derivative.
 
Then, it remains to check how to compute the expansion of the Laurent series solutions and how can we actually compute which inicial conditions are needed.

## Generating the power series solution

Given an equation, we may want to look when it has a "low order" solution. This requires to set up a given ansatz and then extend the possible solution. After that, we need to check the relations between the initial conditions to have an actual solution to the equation.

In [2]:
def generic_solution(equation, gen):
    R = equation.parent()
    C = R.constant_ring()
    order = equation.order(gen)

    nC = C.add_constants(*[f"a_{i}" for i in range(order)])
    a = nC.constant_ring().gens()
    return equation.power_series_solution(gen, {i: a[i] for i in range(order)})

## The linear case

In this case, we have an equation of the shape:
$$L = \alpha_0 u+ \alpha_1 u' + \ldots + \alpha_n u^{(n)}.$$
We assume the elements $\alpha_i \in C[[x]]$, so thei can only have a positive order.

Hence, each term will have order $d - i + ord(\alpha_i)$. This makes comparisons quite trivial and independent of $d$.

In [3]:
C = DifferentialRing(QQ)
R = DifferentialRing(QQ[x], [1])
R.set_constant(C)
F = R.fraction_field()
x, = F.gens()
DO.<u> = DifferentialPolynomialRing(F)
LS.<t> = LaurentSeries(C)
mor = F.laurent_morphism({'x': t}, set_default=True)

### 1.1. A monic case

$$L = u^{(5)} - 1/(1-x) u^{(3)} + 2xu'' - u$$

In [4]:
L = u[5] - (1/(1-x))*u[3] + 2*x*u[2] - u[0]
L

-u_0 + (2*x)*u_2 + ((-1)/(-x + 1))*u_3 + u_5

In [5]:
L.indicial_equation(u, "k", laurent_morph=mor)

((),
 (),
 (A_0*1 + A_1*t + 1/2*A_2*t^2 + 1/6*A_3*t^3 + 1/24*A_4*t^4 + (1/120*A_0 + 1/120*A_3)*t^5 + (1/720*A_1 - 1/360*A_2 + 1/720*A_3 + 1/720*A_4)*t^6 + (1/5040*A_0 + 1/5040*A_2 - 1/5040*A_3 + 1/2520*A_4)*t^7 + (1/13440*A_0 + 1/40320*A_1 - 1/20160*A_2 + 11/40320*A_3 + 1/40320*A_4)*t^8 + (1/72576*A_0 + 1/90720*A_1 - 1/51840*A_2 + 31/362880*A_3 + 31/362880*A_4)*t^9 + (23/1209600*A_0 + 11/3628800*A_1 - 17/3628800*A_2 + 197/3628800*A_3 + 47/1209600*A_4)*t^10 + (401/39916800*A_0 + 131/39916800*A_1 - 3/492800*A_2 + 1/31185*A_3 + 457/19958400*A_4)*t^11 + (139/22809600*A_0 + 907/479001600*A_1 - 1591/479001600*A_2 + 1273/68428800*A_3 + 19/1360800*A_4)*t^12 + (6001/1556755200*A_0 + 13/10886400*A_1 - 13093/6227020800*A_2 + 8147/691891200*A_3 + 7717/889574400*A_4)*t^13 + (218411/87178291200*A_0 + 17051/21794572800*A_1 - 120119/87178291200*A_2 + 167417/21794572800*A_3 + 494807/87178291200*A_4)*t^14 + (245837/145297152000*A_0 + 98401/186810624000*A_1 - 121243/130767436800*A_2 + 27119/5230697472*A_3 + 250451/65383718400*A_4)*t^15 + (124141/105670656000*A_0 + 2551319/6974263296000*A_1 - 1225033/1902071808000*A_2 + 75331589/20922789888000*A_3 + 27823667/10461394944000*A_4)*t^16 + (688343/823350528000*A_0 + 9259661/35568742809600*A_1 - 163021253/355687428096000*A_2 + 33753523/13173608448000*A_3 + 168310481/88921857024000*A_4)*t^17 + (51894811/85364982743040*A_0 + 60597613/320118685286400*A_1 - 2778263/8336424096000*A_2 + 7229237/3880226488320*A_3 + 8811748367/6402373705728000*A_4)*t^18 + (54804348761/121645100408832000*A_0 + 155140291/1105864549171200*A_1 - 20273021/82081714176000*A_2 + 1411431001/1022227734528000*A_3 + 6530384989/6402373705728000*A_4)*t^19 + (452932097/1333827855360000*A_0 + 96241219/910176583680000*A_1 - 452907333169/2432902008176640000*A_2 + 506383371979/486580401635328000*A_3 + 6851294599/8911728967680000*A_4)*t^20 + O(t^21),
  Ideal (0) of Multivariate Polynomial Ring in A_0, A_1, A_2, A_3, A_4, A over Rational Field))

Since we have zero candidates, then we conclude that the order of the series must be greater or equal than 0 and smaller than $5$.

We can check now a generic solution for relations among the initial conditions to be solutions of our equation:

In [6]:
sol = generic_solution(L, u)
mor_sol = L.parent().laurent_morphism(imgs={'u': sol}, constant=sol.parent().constant_ring())
L_eval = mor_sol(L)
L_eval.is_zero()

20

Since we get zero in the evaluation, we can conclude that up to order 20, we have no condition, hence, any possibility for the constants in the generic solution provide a solution to the equation.

This makes sense, because we are considering a linear differential equation, whose solutions are a C-vector space and the initial conditions guarantee a linear independence. 

### 1.2. The special case: Bessel differential equation

In [7]:
L = x^2*u[2] + x*u[1] + (x^2-25)*u[0]
L

(x^2 - 25)*u_0 + x*u_1 + x^2*u_2

In [8]:
L.indicial_equation(u, "k", laurent_morph=mor)

((((-Infinity, +Infinity), [5, -5]),),
 (),
 [(x^2 - 25)*u_0 + x*u_1 + x^2*u_2, 'Cannot-extend'])

In this case, we have an equation so homogeneous that any function with order different than $(0,1)$ may have cancellations. So these candidates are not good enough. Let us dive a bit deeper. If we assume a good order (i.e., different than $0,1$) then we have that 
$$[x^{k}] L\cdot u(x) = k(k-1)u_{k} + ku_{k} - 25u_k = (k^2 - k + k - 25) u_k = (k^2 - 25)u_k.$$
Since we assume the order of $u$ is $k$, then we know that $u_k \neq 0$. Hence we obtain $k \in \{\pm5\}$. These are the integer roots of the actual inidicial polynomial. This case is also quite interesting since extending the initial values is not as simple as it may seem using the classical approach of computing $u''$ from the equation and derivating here.

In this case, we use the following approach:
$$[x^p] L\cdot u(x) = [x^p] x^2u'' + [x^p] xu' + [x^p](x^2-25)u = [x^{p-2}] u'' + [x^{p-1}] u' + [x^{p-2}]u - 25 [x^p]u = p(p-1)u_p + pu_p + u_{p-2} - 25u_p = (p^2 - 25)u_p + u_{p-2},$$
so we can get the value:
$$u_{p} = \frac{u_{p-2}}{(25-p^2)},$$
which is a very simple formula tha works for any $p \notin\{\pm5\}$.

We could easily get this formulation due to the nature of the equation $L$: is is a D-finite equation (linear with polynomial coefficients) so it is known that the sequence of any solution is $P$-finite (or define with a linear recurrence with polynomial coefficients).

In [9]:
def _bessel_coeff(k,n):
    if k < n:
        return 0
    elif k == n:
        return 1/3840
    else:
        return _bessel_coeff(k-2,n)/(n^2 - k^2)
sol = LS.element_class(LS, coefficient_map=lambda k : _bessel_coeff(k,5), order=5)

In [10]:
[sol[i]*factorial(i) for i in range(5,10)]

[1/32, 0, -7/128, 0, 9/128]

In [11]:
f = Bessel(5)(SR('t'))

In [12]:
[f.derivative(i)(t=0) for i in range(20)]

[0,
 0,
 0,
 0,
 0,
 1/32,
 0,
 -7/128,
 0,
 9/128,
 0,
 -165/2048,
 0,
 715/8192,
 0,
 -3003/32768,
 0,
 1547/16384,
 0,
 -12597/131072]

## The non-linear case

Non-linear differential equations are quite common in the context of this repository. A classic example is the $\wp$-Weierstrass function:
$$\wp'^2 = 4\wp^3 + g_2\wp + g_3.$$
Let us check how the indicail method works on this equation:

In [14]:
DO_C = DO.add_constants("g_2", "g_3")
u = DO_C.gen("u")
C_C = DO_C.constant_ring()
F_C = DO_C.base()
x,g_2,g_3 = F_C.gens()
LS_C = LS.add_constants("g_2", "g_3")
t_C = LS_C.gen()
mor = F_C.laurent_morphism({'x': t_C}, set_default=True)

In [15]:
L = u[1]^2 - 4*u[0]^3 - g_2*u[0] - g_3
L

-g_3 - g_2*u_0 - 4*u_0^3 + u_1^2

In [16]:
L.indicial_equation(u, laurent_morph=mor)

((),
 ((-2, -4*u_0^3 + 4*u_0^2),
  (0, 'Small'),
  (1, u_0^2 - g_3),
  (2, 'No-cancellation')),
 ((A_0*1 + O(t^21),
   Ideal (4*A_0^3 + g_2*A_0 + g_3) of Multivariate Polynomial Ring in g_2, g_3, A_0, A over Rational Field),
  (A_0*1 + A_1*t + (3*A_0^2 + 1/4*g_2)*t^2 + 2*A_0*A_1*t^3 + (3*A_0^3 + 1/4*g_2*A_0 + 1/2*A_1^2)*t^4 + (3*A_0^2*A_1 + 3/20*g_2*A_1)*t^5 + (3*A_0^4 + 2/5*g_2*A_0^2 + A_0*A_1^2 + 1/80*g_2^2)*t^6 + (24/7*A_0^3*A_1 + 9/35*g_2*A_0*A_1 + 1/7*A_1^3)*t^7 + (18/7*A_0^5 + 57/140*g_2*A_0^3 + 45/28*A_0^2*A_1^2 + 9/560*g_2^2*A_0 + 33/560*g_2*A_1^2)*t^8 + (25/7*A_0^4*A_1 + 11/28*g_2*A_0^2*A_1 + 5/14*A_0*A_1^3 + 1/120*g_2^2*A_1)*t^9 + (15/7*A_0^6 + 29/70*g_2*A_0^4 + 29/14*A_0^3*A_1^2 + 69/2800*g_2^2*A_0^2 + 37/280*g_2*A_0*A_1^2 + 1/28*A_1^4 + 1/2400*g_2^3)*t^10 + (24/7*A_0^5*A_1 + 186/385*g_2*A_0^3*A_1 + 9/14*A_0^2*A_1^3 + 127/7700*g_2^2*A_0*A_1 + 57/3080*g_2*A_1^3)*t^11 + (12/7*A_0^7 + 148/385*g_2*A_0^5 + 67/28*A_0^4*A_1^2 + 437/15400*g_2^2*A_0^3 + 141/616*g_2*A_0^2*A_1^2 + 3/28*A_0*A_1^4 + 127/184800*g_2^3*A_0 + 1363/369600*g_2^2*A_1^2)*t^12 + (285/91*A_0^6*A_1 + 2175/4004*g_2*A_0^4*A_1 + 85/91*A_0^3*A_1^3 + 2207/80080*g_2^2*A_0^2*A_1 + 207/4004*g_2*A_0*A_1^3 + 3/364*A_1^5 + 7/20800*g_2^3*A_1)*t^13 + (855/637*A_0^8 + 345/1001*g_2*A_0^6 + 1620/637*A_0^5*A_1^2 + 2187/70070*g_2^2*A_0^4 + 4509/14014*g_2*A_0^3*A_1^2 + 555/2548*A_0^2*A_1^4 + 3163/2802800*g_2^3*A_0^2 + 332/35035*g_2^2*A_0*A_1^2 + 579/112112*g_2*A_1^4 + 1/83200*g_2^4)*t^14 + (1752/637*A_0^7*A_1 + 19788/35035*g_2*A_0^5*A_1 + 762/637*A_0^4*A_1^3 + 13087/350350*g_2^2*A_0^3*A_1 + 7123/70070*g_2*A_0^2*A_1^3 + 37/1274*A_0*A_1^5 + 783/1001000*g_2^3*A_0*A_1 + 61/46200*g_2^2*A_1^3)*t^15 + (657/637*A_0^9 + 5961/20020*g_2*A_0^7 + 6495/2548*A_0^6*A_1^2 + 88731/2802800*g_2^2*A_0^5 + 28239/70070*g_2*A_0^4*A_1^2 + 3603/10192*A_0^3*A_1^4 + 40939/28028000*g_2^3*A_0^3 + 111/6160*g_2^2*A_0^2*A_1^2 + 38667/2242240*g_2*A_0*A_1^4 + 37/20384*A_1^6 + 783/32032000*g_2^4*A_0 + 5531/32032000*g_2^3*A_1^2)*t^16 + (1494/637*A_0^8*A_1 + 132705/238238*g_2*A_0^6*A_1 + 891/637*A_0^5*A_1^3 + 1779/38896*g_2^2*A_0^4*A_1 + 76599/476476*g_2*A_0^3*A_1^3 + 675/10192*A_0^2*A_1^5 + 274377/190590400*g_2^3*A_0^2*A_1 + 79071/19059040*g_2^2*A_0*A_1^3 + 50877/38118080*g_2*A_1^5 + 41/3536000*g_2^4*A_1)*t^17 + (498/637*A_0^10 + 29878/119119*g_2*A_0^8 + 1555/637*A_0^7*A_1^2 + 58547/1905904*g_2^2*A_0^6 + 55152/119119*g_2*A_0^5*A_1^2 + 5085/10192*A_0^4*A_1^4 + 45491/25989600*g_2^3*A_0^4 + 1584343/57177120*g_2^2*A_0^3*A_1^2 + 72837/1905904*g_2*A_0^2*A_1^4 + 75/10192*A_0*A_1^6 + 501493/11435424000*g_2^4*A_0^2 + 578273/1143542400*g_2^3*A_0*A_1^2 + 17293/41583360*g_2^2*A_1^4 + 41/127296000*g_2^5)*t^18 + (23640/12103*A_0^9*A_1 + 1191633/2263261*g_2*A_0^7*A_1 + 37025/24206*A_0^6*A_1^3 + 1167157/22632610*g_2^2*A_0^5*A_1 + 4031019/18106088*g_2*A_0^4*A_1^3 + 1440/12103*A_0^3*A_1^5 + 1943023/905304400*g_2^3*A_0^3*A_1 + 124281/13927760*g_2^2*A_0^2*A_1^3 + 187749/36212176*g_2*A_0*A_1^5 + 75/193648*A_1^7 + 34621/1108536000*g_2^4*A_0*A_1 + 153/2173600*g_2^3*A_1^3)*t^19 + (7092/12103*A_0^11 + 4680069/22632610*g_2*A_0^9 + 109197/48412*A_0^8*A_1^2 + 12961107/452652200*g_2^2*A_0^7 + 161457/323323*g_2*A_0^6*A_1^2 + 30855/48412*A_0^5*A_1^4 + 1346203/696388000*g_2^3*A_0^5 + 136233319/3621217600*g_2^2*A_0^4*A_1^2 + 24325911/362121760*g_2*A_0^3*A_1^4 + 7227/387296*A_0^2*A_1^6 + 1426443/22632610000*g_2^4*A_0^3 + 449259/426025600*g_2^3*A_0^2*A_1^2 + 11156337/7242435200*g_2^2*A_0*A_1^4 + 473673/1448487040*g_2*A_1^6 + 34621/44341440000*g_2^5*A_0 + 75833/11085360000*g_2^4*A_1^2)*t^20 + O(t^21),
   Ideal (4*A_0^3 + g_2*A_0 - A_1^2 + g_3) of Multivariate Polynomial Ring in g_2, g_3, A_0, A_1, A over Rational Field)))